## InfluxDB Query Client ##

In [ ]:
from astropy.time import Time, TimeDelta
import rubin_nights.dayobs_utils as rn_dayobs

import rubin_nights.connections as connections
from rubin_nights.influx_query import InfluxQueryClient, AsyncInfluxQueryClient

In [ ]:
# To connect to influx databases where creds are served by repertoire, get credentials from repertoire
#  --- at present this is only the EFD, but we can have a look. 

In [ ]:
# Check out repertoire
import httpx
from rubin_nights.reference_values import API_ENDPOINTS

site = 'usdf'
authtoken = connections.get_access_token("/Users/lynnej/.lsst/usdf_rsp")
auth = ('user', authtoken)

uri = f"{API_ENDPOINTS[site]}/repertoire/discovery/influxdb"
response = httpx.get(uri, auth=auth)
#response.json()

The InfluxQueryClient tries to authenticate with repertoire (which requires an auth token), and if that doesn't work it falls back to segwarides, which is still in operation for now (but is in the process of being phase out). 

In [ ]:
# Full future-style authentication
efd_client = InfluxQueryClient(site='usdf', db_name='efd', repertoire_site='usdf', auth=auth)
efd_client.creds_from

In [ ]:
efd_client.get_topics()[0:3]

In [ ]:
# But if you don't pass the auth token, you cannot access repertoire
efd_client = InfluxQueryClient(site='usdf', db_name='efd', repertoire_site='usdf')
efd_client.creds_from

In [ ]:
efd_client.get_topics()[0:3]

In [ ]:
# And if you try to access a databse which is not in repertoire yet, the fallback to segwarides is still active
pp_client = InfluxQueryClient(site='usdf', db_name='lsst.prompt', repertoire_site='usdf', auth=auth)
pp_client.creds_from

In [ ]:
# Both work ..
pp_client.get_topics()[0:3]

In [ ]:
# Query over a time period
sunset, sunrise = rn_dayobs.day_obs_sunset_sunrise(20260614)

topic = "lsst.sal.Scheduler.logevent_target"
efd_client.select_time_series(topic, '*', sunset, sunrise, index=1).head()

In [ ]:
# Query the last N entries before a time 
efd_client.select_top_n(topic, '*', 3, time_cut=sunset, index=1)

In [ ]:
# or an arbitrary query
query = f'select * from "lsst.sal.Scheduler.logevent_target"'
query += f"where time >= '{sunset.utc.isot}Z' and time <= '{sunrise.utc.isot}Z' and targetName != 'lowdust' and salIndex = 1"
efd_client.query(query).head()

# Let's look briefly at the sync vs. async clients -- above were the sync clients.

The async client has the same api, but needs to be awaited.

In [ ]:
async_efd = AsyncInfluxQueryClient(site='usdf', db_name='efd', repertoire_site='usdf', auth=auth)
async_efd.creds_from

In [ ]:
topics = await async_efd.get_topics()
topics[0:3]

In [ ]:
# Query over a time period
sunset, sunrise = rn_dayobs.day_obs_sunset_sunrise(20260614)

topic = "lsst.sal.Scheduler.logevent_target"
results = await async_efd.select_time_series(topic, '*', sunset, sunrise, index=1)
results.head()

In [ ]:
# Query the last N entries before a time 
await async_efd.select_top_n(topic, '*', 3, time_cut=sunset, index=1)

In [ ]:
# or an arbitrary query
query = f'select * from "lsst.sal.Scheduler.logevent_target"'
query += f"where time >= '{sunset.utc.isot}Z' and time <= '{sunrise.utc.isot}Z' and targetName != 'lowdust' and salIndex = 1"
results = await async_efd.query(query)
results.head()

# Using in context managers 

To close the connections and free the pool after use, either use in a context or call close/aclose directly. 

In [ ]:
with InfluxQueryClient(site='usdf', db_name='efd', repertoire_site='usdf', auth=auth) as efdq:
    result = efdq.query(query)

print(efdq.httpx_client.is_closed)
result.head()

In [ ]:
async with AsyncInfluxQueryClient(site='usdf', db_name='efd', repertoire_site='usdf', auth=auth) as efdq:
    result = await efdq.query(query)

print(efdq.async_client.is_closed)
result.head()

In [ ]:
# Or directly closing yourself 
efdq = InfluxQueryClient(site='usdf', db_name='efd', repertoire_site='usdf', auth=auth)
print('before closing connection', efdq.httpx_client.is_closed)
efdq.close()
print('after closing connection', efdq.httpx_client.is_closed)